<a href="https://colab.research.google.com/github/Thilac01/Statistical-Learning-e22395/blob/main/Statistical_Learning_Assignement7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1. Prior Belief Boundaries**

The initial prior represents the engineering confidence in the component's structural integrity before any sensor data is collected.

For a Beta distribution $\Theta \sim \text{Beta}(\alpha, \beta)$, the expected value is calculated analytically as:

$$ \mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} $$

Given $\alpha = 8$ and $\beta = 1.5$:

$$ \mathbb{E}[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.8421 $$

**Why this prior is appropriate:**
A Beta distribution is naturally bounded on the interval $(0, 1)$, matching the physical limits of the remaining stiffness factor. Setting $\alpha > \beta$ with a high expected value ($\sim$84%) models an optimistic but realistic engineering assumption: the component is highly likely to be healthy (close to 1.0) post-manufacturing, but it retains a left-skewed "tail" that assigns non-zero probability to pre-existing manufacturing defects or early damage.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define the bounded domain (avoiding absolute zero to prevent log(0) issues later)
theta_grid = np.linspace(0.01, 1.0, 500)

# Calculate Beta(8, 1.5) density
prior_density = beta.pdf(theta_grid, 8, 1.5)

# Plot using Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior_density,
                         mode='lines',
                         name='Beta(8, 1.5)',
                         line=dict(color='blue', width=3)))

fig.update_layout(title='Initial Prior Belief: Beta(8, 1.5)',
                  xaxis_title='Stiffness Efficiency Factor (θ)',
                  yaxis_title='Probability Density',
                  template='plotly_white')
fig.show()

### **2. Structural Likelihood Formulation**

The structural measurement model is given by $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$.
Taking the natural logarithm of both sides yields:

$$ \ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k $$

Since $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the logged measurement $\ln(y_k)$ follows a normal distribution centered at $\ln(\theta K_{\text{nominal}})$. Therefore, $y_k$ itself follows a Log-Normal distribution. The likelihood contribution of a single continuous sensor measurement $y_k$ is:

$$ L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln(y_k) - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right) $$

Assuming the sequential sensor noise terms $\epsilon_k$ are independent and identically distributed, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of individual likelihoods:

$$ L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln(y_i) - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right) $$

---

### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

A conjugate prior would produce a posterior distribution belonging to the same probability distribution family. However, the combination of a Beta prior (with polynomial terms $\theta^{\alpha-1}(1-\theta)^{\beta-1}$) and a Log-Normal likelihood (with $\ln(\theta)$ inside an exponential squared term) cannot be algebraically manipulated into the form of another Beta distribution.

Because an exact closed-form analytical solution does not exist, the sequential update must be processed recursively up to a proportionality constant:

$$ f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) $$

where the initial condition at step $k=0$ is $f_{\Theta \mid \mathbf{Y}^{(0)}}(\theta) = f_{\Theta}^{(0)}(\theta)$.

---

### **4. Running Point Estimates**

Since we lack a closed-form posterior probability density function, the point estimates must be defined as definite integrals over the domain $(0, 1]$.

**Running Posterior Mean (Bayesian Minimum Mean Square Error estimator):**
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta $$

**Running Maximum A Posteriori (MAP) estimator:**
$$ \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \underset{\theta \in (0, 1]}{\text{arg max}} \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) $$

---

### **5. Algorithmic Grid Approximation and Normalization**

To maintain and update this distribution computationally, we implement a discrete grid approximation:

1. **Grid Initialization:** Discretize the domain into an array of $N$ equally spaced values from a lower bound just above zero (e.g., $0.01$) up to $1.0$. The lower bound avoids undefined $\ln(0)$ operations in the likelihood. Evaluate the Beta PDF across this grid to establish the $k=0$ prior array.
2. **Likelihood Evaluation:** When reading $y_k$ arrives, compute the Log-Normal likelihood array for $y_k$ across all values in the discretized $\theta$ grid.
3. **Pointwise Multiplication:** Multiply the unnormalized likelihood array pointwise with the posterior array from step $k-1$ to get the unnormalized posterior for step $k$.
4. **Trapezoidal Normalization:** Integrate the unnormalized posterior array over the $\theta$ grid using the trapezoidal rule (`np.trapezoid`). The result is the total area (the marginal likelihood constant). Divide the unnormalized posterior array by this area to enforce $\int f(\theta) d\theta = 1$.
5. **Estimate Extraction:** Calculate the MAP by finding the $\theta$ grid value at the array's maximum index. Calculate the Mean by performing a trapezoidal integration over `theta_grid * normalized_posterior`.

---

### **6. Performance Tracking and Degradation Convergence Analysis**

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Set random seed for reproducible simulation
np.random.seed(42)

# System Parameters
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_steps = 15

# 1. Simulate Sensor Stream
epsilons = np.random.normal(0, sigma, n_steps)
y_obs = theta_true * K_nominal * np.exp(epsilons)

# 2. Grid Setup
theta_grid = np.linspace(0.01, 1.0, 1000)
prior = beta.pdf(theta_grid, 8, 1.5)

# Storage Dictionaries/Lists
posteriors_history = {0: prior.copy()}
mean_estimates = [np.trapezoid(theta_grid * prior, x=theta_grid)]
map_estimates = [theta_grid[np.argmax(prior)]]

current_posterior = prior.copy()

# 3. Track Estimators (Sequential Update)
for k in range(1, n_steps + 1):
    y_k = y_obs[k-1]

    # Calculate likelihood across the grid
    # Constants 1/(y*sigma*sqrt(2pi)) scale out during normalization, so we focus on the exponent
    log_y = np.log(y_k)
    log_theta_K = np.log(theta_grid * K_nominal)
    likelihood = np.exp(-0.5 * ((log_y - log_theta_K) / sigma)**2)

    # Pointwise update
    unnormalized = current_posterior * likelihood

    # Normalize via trapezoidal integration
    area = np.trapezoid(unnormalized, x=theta_grid)
    current_posterior = unnormalized / area

    # Store milestones
    if k in [1, 2, 5, 10, 15]:
        posteriors_history[k] = current_posterior.copy()

    # Calculate and store estimates
    mean_est = np.trapezoid(theta_grid * current_posterior, x=theta_grid)
    map_est = theta_grid[np.argmax(current_posterior)]

    mean_estimates.append(mean_est)
    map_estimates.append(map_est)

# 4. Visualize Curves & Timeline
# Plot 1: Full Density Curves
fig1 = go.Figure()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
milestones = [0, 1, 2, 5, 10, 15]

for idx, step in enumerate(milestones):
    fig1.add_trace(go.Scatter(x=theta_grid, y=posteriors_history[step],
                              mode='lines',
                              name=f'Step {step} Posterior',
                              line=dict(color=colors[idx])))

fig1.add_vline(x=theta_true, line_dash="dash", line_color="black",
               annotation_text="True Damage (0.68)", annotation_position="top left")

fig1.update_layout(title='Evolution of Structural Stiffness Density (Over 15 Readings)',
                   xaxis_title='Stiffness Efficiency Factor (θ)',
                   yaxis_title='Probability Density',
                   template='plotly_white')
fig1.show()

# Plot 2: Estimator Convergence
fig2 = go.Figure()
steps_arr = np.arange(0, n_steps + 1)

fig2.add_trace(go.Scatter(x=steps_arr, y=mean_estimates,
                          mode='lines+markers', name='Bayesian Mean'))
fig2.add_trace(go.Scatter(x=steps_arr, y=map_estimates,
                          mode='lines+markers', name='MAP Estimate'))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="black",
               annotation_text="Ground Truth (0.68)", annotation_position="bottom right")

fig2.update_layout(title='Convergence of Estimators vs. Inspection Milestones',
                   xaxis_title='Inspection Step (k)',
                   yaxis_title='Estimated θ',
                   template='plotly_white',
                   yaxis=dict(range=[0.5, 1.0]))
fig2.show()

### **System Behavior and Convergence Analysis**

*   **Overcoming the Prior:** The initial state (Step 0) firmly anchors expectations around $\theta \approx 0.84$. However, because Bayesian updates inherently penalize predictions that heavily contradict new evidence, the density distribution rapidly shifts leftward. By roughly the 4th to 6th sensor reading, both the MAP and the Bayesian Mean estimators overcome the optimistic prior and confidently isolate the true 68% degradation state.
*   **Narrowing Density (Variance Reduction):** As $k$ increments to 10 and 15, the density curve aggressively narrows into a highly concentrated peak around $0.68$. In safety-critical aerospace and civil engineering contexts, this narrowing reduces the uncertainty envelope. Tighter variance implies that engineers don't have to rely on ultra-conservative, worst-case maintenance schedules; instead, they have mathematical justification to pinpoint structural thresholds and safely maximize the remaining operational life of the component.